# Healthcare Claims Analysis
## January – April 2026

This notebook provides a professional analysis of medical claims data, focusing on:
- Service type and benefit breakdowns
- Age bracket analysis (with service type averages)
- Provider intelligence (hospital spending patterns)
- Monthly trends
- Patient switching behaviour and hospital retention
- Same‑day OP‑IP transitions

All analysis is based on the `VISIT_KEY` concept: one member + one arrival date + one service type = one unique visit.

## 1. Setup and Data Loading

In [15]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

# Load data
df = pd.read_csv('../data/visit_for_jan_to_end_of_May.csv')

# Standardise column names (strip spaces)
df.columns = df.columns.str.strip()

# Convert date columns to datetime
df['ARRIVAL DATE'] = pd.to_datetime(df['ARRIVAL DATE'])
df['TRANSACTION DATE'] = pd.to_datetime(df['TRANSACTION DATE'])
df['DOB'] = pd.to_datetime(df['DOB'], errors='coerce')

# Convert amount to numeric (coerce errors)
df['AMOUNT'] = pd.to_numeric(df['AMOUNT'], errors='coerce')

print(f"Dataset shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head()

Dataset shape: 28,478 rows, 44 columns


,EDI_NO,CLAIM ID,CENTRAL ID,CLAIM TYPE,SCHEME,MEMBER NUMBER,INTEG MEMBER NUMBER,OTHER NUMBER,OFFICE BRANCH,CARD SERIAL,...,MEMBER SPLIT AMOUNT,GLOBAL INVOICE NUMBER,PRINCIPAL NAMES,PRINCIPAL MEMBER NUMBER,PRINCIPAL OTHER NUMBER,RELATIONSHIP TO PRINCIPAL,PHONE NUMBER,PHONE NUMBER2,PARENT POOL,PARENT POOL DESCRIPTION
0,NaN,469354248,387361511,Normal claim,DEFENCE FORCES MEDICAL INSURANCE SCHEME,DEFMIS-105377-01,2009664.0,105377,NaN,VCKE0001643243,...,0,NaN,NaN,NaN,NaN,NaN,2.547250e+11,NaN,NaN,NaN
1,NaN,469349400,387354913,Normal claim,DEFENCE FORCES MEDICAL INSURANCE SCHEME,DEFMIS-105377-01,2009664.0,105377,NaN,VCKE0001643243,...,0,NaN,NaN,NaN,NaN,NaN,2.547250e+11,NaN,NaN,NaN
2,NaN,470806257,404027391,Normal claim,DEFENCE FORCES MEDICAL INSURANCE SCHEME,DEFMIS-105377-01,2009664.0,105377,NaN,VCKE0001643243,...,0,NaN,NaN,NaN,NaN,NaN,2.547250e+11,NaN,NaN,NaN
3,NaN,470823825,404138004,Normal claim,DEFENCE FORCES MEDICAL INSURANCE SCHEME,DEFMIS-105377-01,2009664.0,105377,NaN,VCKE0001643243,...,0,NaN,NaN,NaN,NaN,NaN,2.547250e+11,NaN,NaN,NaN
4,NaN,470805121,404024238,Normal claim,DEFENCE FORCES MEDICAL INSURANCE SCHEME,DEFMIS-105377-01,2009664.0,105377,NaN,VCKE0001643243,...,0,NaN,NaN,NaN,NaN,NaN,2.547250e+11,NaN,NaN,NaN


## 2. Create Visit Key

A visit is defined as a unique combination of member number, arrival date, and service type.

In [16]:
df['VISIT_KEY'] = (
    df['MEMBER NUMBER'].astype(str) + '_' +
    df['ARRIVAL DATE'].dt.strftime('%Y-%m-%d') + '_' +
    df['SERVICE TYPE'].astype(str)
)

print("VISIT_KEY created. Example:", df['VISIT_KEY'].iloc[0])

VISIT_KEY created. Example: DEFMIS-105377-01_2026-01-23_OP


## 3. High‑Level Summary by Service Type

In [17]:
# Aggregate by SERVICE TYPE
service_summary = df.groupby('SERVICE TYPE').agg(
    TOTAL_AMOUNT=('AMOUNT', 'sum'),
    UNIQUE_VISITS=('VISIT_KEY', 'nunique')
).reset_index()

# Add percentage and average cost per visit
grand_total = service_summary['TOTAL_AMOUNT'].sum()
service_summary['SHARE (%)'] = (service_summary['TOTAL_AMOUNT'] / grand_total * 100).round(2)
service_summary['AVG_COST_PER_VISIT'] = (service_summary['TOTAL_AMOUNT'] / service_summary['UNIQUE_VISITS']).round(2)

# Format for display with thousands separators
display_summary = service_summary.copy()
display_summary['TOTAL_AMOUNT'] = display_summary['TOTAL_AMOUNT'].apply(lambda x: f"Ksh {x:,.0f}")
display_summary['AVG_COST_PER_VISIT'] = display_summary['AVG_COST_PER_VISIT'].apply(lambda x: f"Ksh {x:,.2f}")
display_summary['UNIQUE_VISITS'] = display_summary['UNIQUE_VISITS'].apply(lambda x: f"{x:,}")
display_summary['SHARE (%)'] = display_summary['SHARE (%)'].astype(str) + '%'

print("### Service Type Summary\n")
display(display_summary)

# Bar chart
fig = px.bar(
    service_summary,
    x='SERVICE TYPE',
    y='TOTAL_AMOUNT',
    title='Total Amount by Service Type',
    text=service_summary['TOTAL_AMOUNT'].apply(lambda x: f'{x/1e6:.1f}M'),
    color='TOTAL_AMOUNT',
    color_continuous_scale='Viridis'
)
fig.update_traces(textposition='outside')
fig.update_layout(yaxis_tickformat=',.0f', height=450)
fig.show()

### Service Type Summary



,SERVICE TYPE,TOTAL_AMOUNT,UNIQUE_VISITS,SHARE (%),AVG_COST_PER_VISIT
0,IP,"Ksh 240,342,238","1,809",57.83%,"Ksh 132,859.17"
1,OP,"Ksh 175,262,409","22,797",42.17%,"Ksh 7,687.96"


## 4. Breakdown by Benefit Description

In [18]:
benefit_summary = df.groupby('BENEFIT DESC').agg(
    UNIQUE_VISITS=('VISIT_KEY', 'nunique'),
    TOTAL_AMOUNT=('AMOUNT', 'sum')
).reset_index()

benefit_summary['AVG_COST_PER_VISIT'] = (benefit_summary['TOTAL_AMOUNT'] / benefit_summary['UNIQUE_VISITS']).round(2)
benefit_summary = benefit_summary.sort_values('TOTAL_AMOUNT', ascending=False)

# Format with commas
benefit_summary['TOTAL_AMOUNT'] = benefit_summary['TOTAL_AMOUNT'].apply(lambda x: f"Ksh {x:,.2f}")
benefit_summary['AVG_COST_PER_VISIT'] = benefit_summary['AVG_COST_PER_VISIT'].apply(lambda x: f"Ksh {x:,.2f}")
benefit_summary['UNIQUE_VISITS'] = benefit_summary['UNIQUE_VISITS'].apply(lambda x: f"{x:,}")

print("### Benefit Description Summary\n")
display(benefit_summary)

### Benefit Description Summary



,BENEFIT DESC,UNIQUE_VISITS,TOTAL_AMOUNT,AVG_COST_PER_VISIT
0,IN PATIENT OVERALL / HOSPITALIZATION/ACCOMODATION,"1,809","Ksh 240,342,237.70","Ksh 132,859.17"
3,OUT PATIENT OVERALL,"21,739","Ksh 162,732,907.14","Ksh 7,485.76"
2,OUT PATIENT OPTICAL,804,"Ksh 8,521,099.23","Ksh 10,598.38"
1,OUT PATIENT DENTAL,534,"Ksh 4,008,402.14","Ksh 7,506.37"


## 5. Age Bracket Analysis

We calculate age at the time of visit using the patient's date of birth and the arrival date. Then we group into age brackets and analyse visit counts, member counts, and total spend.

In [19]:
# Calculate age at visit (in years)
df['AGE_AT_VISIT'] = (df['ARRIVAL DATE'] - df['DOB']).dt.days // 365
# Cap at reasonable maximum
df['AGE_AT_VISIT'] = df['AGE_AT_VISIT'].clip(0, 120)

# Define age brackets
bins = [0, 21, 40, 50, 60, 79, 200]
labels = ['0-21', '22-40', '41-50', '51-60', '61-79', '80+']
df['AGE_BRACKET'] = pd.cut(df['AGE_AT_VISIT'], bins=bins, labels=labels, right=True)

# Aggregate
age_summary = df.groupby('AGE_BRACKET').agg(
    UNIQUE_VISITS=('VISIT_KEY', 'nunique'),
    UNIQUE_MEMBERS=('MEMBER NUMBER', 'nunique'),
    TOTAL_AMOUNT=('AMOUNT', 'sum')
).reset_index()

# Add percentage of total amount
total_amount_all = age_summary['TOTAL_AMOUNT'].sum()
age_summary['SHARE (%)'] = (age_summary['TOTAL_AMOUNT'] / total_amount_all * 100).round(2)

# Format for display
age_display = age_summary.copy()
age_display['UNIQUE_VISITS'] = age_display['UNIQUE_VISITS'].apply(lambda x: f"{x:,}")
age_display['UNIQUE_MEMBERS'] = age_display['UNIQUE_MEMBERS'].apply(lambda x: f"{x:,}")
age_display['TOTAL_AMOUNT'] = age_display['TOTAL_AMOUNT'].apply(lambda x: f"Ksh {x:,.0f}")
age_display['SHARE (%)'] = age_display['SHARE (%)'].astype(str) + '%'

print("### Age Bracket Summary\n")
display(age_display)

# Identify highest values
max_visits_bracket = age_summary.loc[age_summary['UNIQUE_VISITS'].idxmax(), 'AGE_BRACKET']
max_members_bracket = age_summary.loc[age_summary['UNIQUE_MEMBERS'].idxmax(), 'AGE_BRACKET']
max_amount_bracket = age_summary.loc[age_summary['TOTAL_AMOUNT'].idxmax(), 'AGE_BRACKET']

print("\n### Highest Categories")
print(f"- **HIGHEST VISITS**: {max_visits_bracket}")
print(f"- **HIGHEST MEMBERS**: {max_members_bracket}")
print(f"- **HIGHEST AMOUNT**: {max_amount_bracket}")

# Bar chart for amount by age bracket
fig_age = px.bar(
    age_summary,
    x='AGE_BRACKET',
    y='TOTAL_AMOUNT',
    title='Total Amount by Age Bracket',
    text=age_summary['TOTAL_AMOUNT'].apply(lambda x: f'{x/1e6:.1f}M'),
    color='TOTAL_AMOUNT',
    color_continuous_scale='Blues'
)
fig_age.update_traces(textposition='outside')
fig_age.update_layout(yaxis_tickformat=',.0f', height=450)
fig_age.show()

### Age Bracket Summary



,AGE_BRACKET,UNIQUE_VISITS,UNIQUE_MEMBERS,TOTAL_AMOUNT,SHARE (%)
0,0-21,"2,641","1,427","Ksh 27,793,939",6.69%
1,22-40,698,332,"Ksh 8,583,336",2.07%
2,41-50,"1,422",534,"Ksh 22,955,025",5.52%
3,51-60,"5,408","1,786","Ksh 88,698,666",21.34%
4,61-79,"14,224","3,502","Ksh 261,494,716",62.92%
5,80+,209,30,"Ksh 6,057,845",1.46%



### Highest Categories
- **HIGHEST VISITS**: 61-79
- **HIGHEST MEMBERS**: 61-79
- **HIGHEST AMOUNT**: 61-79


## 6. Age Bracket × Service Type Analysis (with Averages)

For each age bracket, we break down by service type (OP / IP) to see:
- Total amount, unique visits, unique members
- Average cost per visit
- Average cost per member

This helps identify which age groups drive high costs per service type.

In [20]:
# Group by age bracket and service type
age_service = df.groupby(['AGE_BRACKET', 'SERVICE TYPE']).agg(
    TOTAL_AMOUNT=('AMOUNT', 'sum'),
    UNIQUE_VISITS=('VISIT_KEY', 'nunique'),
    UNIQUE_MEMBERS=('MEMBER NUMBER', 'nunique')
).reset_index()

# Compute averages
age_service['AVG_COST_PER_VISIT'] = age_service['TOTAL_AMOUNT'] / age_service['UNIQUE_VISITS']
age_service['AVG_COST_PER_MEMBER'] = age_service['TOTAL_AMOUNT'] / age_service['UNIQUE_MEMBERS']

# Sort for meaningful display
age_service = age_service.sort_values(['AGE_BRACKET', 'SERVICE TYPE'])

# Create a pivot table style display
# We'll create two views: one for amount/visits and one for averages
pivot_amount = age_service.pivot(index='AGE_BRACKET', columns='SERVICE TYPE', values='TOTAL_AMOUNT').fillna(0)
pivot_visits = age_service.pivot(index='AGE_BRACKET', columns='SERVICE TYPE', values='UNIQUE_VISITS').fillna(0)
pivot_members = age_service.pivot(index='AGE_BRACKET', columns='SERVICE TYPE', values='UNIQUE_MEMBERS').fillna(0)
pivot_avg_visit = age_service.pivot(index='AGE_BRACKET', columns='SERVICE TYPE', values='AVG_COST_PER_VISIT').fillna(0)
pivot_avg_member = age_service.pivot(index='AGE_BRACKET', columns='SERVICE TYPE', values='AVG_COST_PER_MEMBER').fillna(0)

# Format for display
def format_ksh(x):
    return f"Ksh {x:,.0f}"

def format_number(x):
    return f"{x:,.0f}"

print("### Total Amount by Age Bracket and Service Type\n")
display(pivot_amount.applymap(format_ksh))

print("\n### Unique Visits by Age Bracket and Service Type\n")
display(pivot_visits.applymap(format_number))

print("\n### Average Cost per Visit (Ksh) by Age Bracket and Service Type\n")
display(pivot_avg_visit.applymap(lambda x: f"Ksh {x:,.2f}"))

print("\n### Average Cost per Member (Ksh) by Age Bracket and Service Type\n")
display(pivot_avg_member.applymap(lambda x: f"Ksh {x:,.2f}"))

# Visualisation: Stacked bar chart of amount by age bracket, split by service type
fig_age_service = px.bar(
    age_service,
    x='AGE_BRACKET',
    y='TOTAL_AMOUNT',
    color='SERVICE TYPE',
    title='Total Amount by Age Bracket and Service Type',
    text=age_service['TOTAL_AMOUNT'].apply(lambda x: f'{x/1e6:.1f}M'),
    barmode='stack'
)
fig_age_service.update_traces(textposition='inside')
fig_age_service.update_layout(yaxis_tickformat=',.0f', height=500)
fig_age_service.show()

# Bar chart for average cost per visit by age bracket
fig_avg_visit = px.bar(
    age_service,
    x='AGE_BRACKET',
    y='AVG_COST_PER_VISIT',
    color='SERVICE TYPE',
    title='Average Cost per Visit by Age Bracket and Service Type',
    barmode='group',
    text=age_service['AVG_COST_PER_VISIT'].apply(lambda x: f'Ksh {x:,.0f}')
)
fig_avg_visit.update_traces(textposition='outside')
fig_avg_visit.update_layout(yaxis_tickformat=',.0f', height=500)
fig_avg_visit.show()

### Total Amount by Age Bracket and Service Type



SERVICE TYPE,IP,OP
AGE_BRACKET,,
0-21,"Ksh 14,945,773","Ksh 12,848,166"
22-40,"Ksh 4,401,944","Ksh 4,181,392"
41-50,"Ksh 13,623,865","Ksh 9,331,160"
51-60,"Ksh 50,475,437","Ksh 38,223,229"
61-79,"Ksh 153,023,543","Ksh 108,471,173"
80+,"Ksh 3,871,676","Ksh 2,186,169"



### Unique Visits by Age Bracket and Service Type



SERVICE TYPE,IP,OP
AGE_BRACKET,,
0-21,107,"2,534"
22-40,25,673
41-50,74,"1,348"
51-60,314,"5,094"
61-79,"1,271","12,953"
80+,18,191



### Average Cost per Visit (Ksh) by Age Bracket and Service Type



SERVICE TYPE,IP,OP
AGE_BRACKET,,
0-21,"Ksh 139,680.12","Ksh 5,070.31"
22-40,"Ksh 176,077.76","Ksh 6,213.06"
41-50,"Ksh 184,106.28","Ksh 6,922.23"
51-60,"Ksh 160,749.80","Ksh 7,503.58"
61-79,"Ksh 120,396.18","Ksh 8,374.21"
80+,"Ksh 215,093.10","Ksh 11,445.91"



### Average Cost per Member (Ksh) by Age Bracket and Service Type



SERVICE TYPE,IP,OP
AGE_BRACKET,,
0-21,"Ksh 171,790.50","Ksh 9,144.60"
22-40,"Ksh 200,088.37","Ksh 12,787.13"
41-50,"Ksh 247,706.63","Ksh 17,605.96"
51-60,"Ksh 286,792.26","Ksh 21,668.50"
61-79,"Ksh 294,843.05","Ksh 31,304.81"
80+,"Ksh 483,959.47","Ksh 72,872.32"


## 7. Provider Intelligence (Hospital × Benefit Matrix)

Shows total amount spent per hospital per benefit category.

In [21]:
# Pivot table: Hospital x Benefit Desc
provider_benefit = pd.pivot_table(
    df,
    index='MAIN HOSPITAL',
    columns='BENEFIT DESC',
    values='AMOUNT',
    aggfunc='sum',
    fill_value=0
)

# Add total column and sort
provider_benefit['TOTAL_COST'] = provider_benefit.sum(axis=1)
provider_benefit = provider_benefit.sort_values('TOTAL_COST', ascending=False)

# Display with formatted numbers (commas, 2 decimals)
provider_benefit_display = provider_benefit.copy()
for col in provider_benefit_display.columns:
    if col != 'TOTAL_COST':
        provider_benefit_display[col] = provider_benefit_display[col].apply(lambda x: f"{x:,.2f}")
    else:
        provider_benefit_display[col] = provider_benefit_display[col].apply(lambda x: f"Ksh {x:,.2f}")

print("### Top 10 Hospitals by Total Spend\n")
display(provider_benefit_display.head(10))

### Top 10 Hospitals by Total Spend



BENEFIT DESC,IN PATIENT OVERALL / HOSPITALIZATION/ACCOMODATION,OUT PATIENT DENTAL,OUT PATIENT OPTICAL,OUT PATIENT OVERALL,TOTAL_COST
MAIN HOSPITAL,,,,,
ULINZI PRIME HEALTH SERVICES FUND (UPHSF),"28,088,608.82","957,448.93","38,960.32","40,058,066.28","Ksh 69,143,084.35"
NAIROBI WEST HOSP,"20,427,176.53","84,789.00","9,000.00","4,628,231.87","Ksh 25,149,197.40"
NAIROBI HOSP REFERRAL,"16,133,177.92","62,489.00",0.00,"8,218,279.33","Ksh 24,413,946.25"
THE AGA KHAN HOSP KIS,"14,036,897.48","61,667.73","96,375.00","5,429,911.47","Ksh 19,624,851.68"
ST LUKE ORTHOPEADIC ELD,"15,658,653.49","63,966.00","32,566.00","3,557,172.57","Ksh 19,312,358.06"
THE KAREN HOSP REFERRAL,"14,335,407.00","50,657.00","21,000.00","4,698,392.60","Ksh 19,105,456.60"
MONALIFE PHARMACEUTICALS LTD,0.00,0.00,0.00,"18,611,238.60","Ksh 18,611,238.60"
NAIROBI SOUTH HOSPITAL,"13,851,222.69","45,600.41",0.00,"705,606.87","Ksh 14,602,429.97"
THE MATER MISERICORDIAE HOSPITAL,"10,738,630.42","44,077.00",0.00,"2,943,667.59","Ksh 13,726,375.01"


## 8. Monthly Trends

In [22]:
# Create month-period column
df['YEAR_MONTH'] = df['TRANSACTION DATE'].dt.to_period('M')

# Aggregate monthly
monthly = df.groupby('YEAR_MONTH').agg(
    TOTAL_AMOUNT=('AMOUNT', 'sum'),
    TOTAL_CLAIMS=('CLAIM ID', 'count'),
    UNIQUE_VISITS=('VISIT_KEY', 'nunique')
).reset_index()

monthly['YEAR_MONTH_STR'] = monthly['YEAR_MONTH'].astype(str)
monthly = monthly.sort_values('YEAR_MONTH')

# Format with commas
monthly_display = monthly.copy()
monthly_display['TOTAL_AMOUNT'] = monthly_display['TOTAL_AMOUNT'].apply(lambda x: f"Ksh {x:,.0f}")
monthly_display['TOTAL_CLAIMS'] = monthly_display['TOTAL_CLAIMS'].apply(lambda x: f"{x:,}")
monthly_display['UNIQUE_VISITS'] = monthly_display['UNIQUE_VISITS'].apply(lambda x: f"{x:,}")

print("### Monthly Summary\n")
display(monthly_display[['YEAR_MONTH_STR', 'TOTAL_AMOUNT', 'TOTAL_CLAIMS', 'UNIQUE_VISITS']])

# Line chart (dual axis) – uses raw numbers
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(x=monthly['YEAR_MONTH_STR'], y=monthly['TOTAL_AMOUNT'],
                         name='Total Amount (Ksh)', mode='lines+markers'),
              secondary_y=False)
fig.add_trace(go.Scatter(x=monthly['YEAR_MONTH_STR'], y=monthly['UNIQUE_VISITS'],
                         name='Unique Visits', mode='lines+markers'),
              secondary_y=True)

fig.update_layout(title='Monthly Trends: Amount vs Unique Visits', height=450)
fig.update_yaxes(title_text='Amount (Ksh)', secondary_y=False)
fig.update_yaxes(title_text='Number of Visits', secondary_y=True)
fig.show()

### Monthly Summary



,YEAR_MONTH_STR,TOTAL_AMOUNT,TOTAL_CLAIMS,UNIQUE_VISITS
0,2026-01,"Ksh 75,156,783","5,936","5,083"
1,2026-02,"Ksh 89,102,713","5,594","4,836"
2,2026-03,"Ksh 86,890,500","5,754","4,980"
3,2026-04,"Ksh 75,124,285","5,670","4,946"
4,2026-05,"Ksh 89,330,366","5,524","4,783"


## 9. Patient Switching Analysis

We analyse patients who **start** at one hospital and later visit a **different** hospital. The analysis identifies:
- Hospitals that lose the most patients (worst retention)
- Where those patients go (flow analysis)
- Retention rates based on first vs last hospital

In [23]:
# Helper functions to get hospital at min/max date
def get_first_hospital(group):
    return group.loc[group['ARRIVAL DATE'].idxmin(), 'MAIN HOSPITAL']

def get_last_hospital(group):
    return group.loc[group['ARRIVAL DATE'].idxmax(), 'MAIN HOSPITAL']

# Member-level summary
member_first = df.groupby('MEMBER NUMBER').apply(get_first_hospital).reset_index(name='FIRST_HOSPITAL')
member_last = df.groupby('MEMBER NUMBER').apply(get_last_hospital).reset_index(name='LAST_HOSPITAL')
member_visits = df.groupby('MEMBER NUMBER')['CLAIM ID'].count().reset_index(name='VISIT_COUNT')
member_dates = df.groupby('MEMBER NUMBER')['ARRIVAL DATE'].agg(['min', 'max']).reset_index()
member_dates.columns = ['MEMBER NUMBER', 'FIRST_DATE', 'LAST_DATE']

# Merge
member_summary = member_first.merge(member_last, on='MEMBER NUMBER')
member_summary = member_summary.merge(member_visits, on='MEMBER NUMBER')
member_summary = member_summary.merge(member_dates, on='MEMBER NUMBER')
member_summary['SWITCHED'] = member_summary['FIRST_HOSPITAL'] != member_summary['LAST_HOSPITAL']

# Aggregate by first hospital
hospital_switch = member_summary.groupby('FIRST_HOSPITAL').agg(
    PATIENTS_STARTED=('MEMBER NUMBER', 'nunique'),
    PATIENTS_LOST=('SWITCHED', 'sum'),
    TOTAL_VISITS=('VISIT_COUNT', 'sum')
).reset_index()

hospital_switch['RETENTION_RATE'] = (1 - hospital_switch['PATIENTS_LOST'] / hospital_switch['PATIENTS_STARTED']) * 100
hospital_switch = hospital_switch.sort_values('PATIENTS_LOST', ascending=False)

print("### Worst Hospitals (Most Patients Lost)\n")
worst_display = hospital_switch.head(10)[['FIRST_HOSPITAL', 'PATIENTS_STARTED', 'PATIENTS_LOST', 'RETENTION_RATE']].copy()
worst_display.columns = ['Hospital', 'Patients Started', 'Patients Lost', 'Retention Rate (%)']
worst_display['Patients Started'] = worst_display['Patients Started'].apply(lambda x: f"{x:,}")
worst_display['Patients Lost'] = worst_display['Patients Lost'].apply(lambda x: f"{x:,}")
worst_display['Retention Rate (%)'] = worst_display['Retention Rate (%)'].round(1)
display(worst_display)

print("### Best Hospitals (Most Patients Retained)\n")
best_display = hospital_switch.sort_values('PATIENTS_STARTED', ascending=False).head(10).copy()
best_display = best_display[['FIRST_HOSPITAL', 'PATIENTS_STARTED', 'PATIENTS_LOST', 'RETENTION_RATE']]
best_display.columns = ['Hospital', 'Patients Started', 'Patients Lost', 'Retention Rate (%)']
best_display['Patients Started'] = best_display['Patients Started'].apply(lambda x: f"{x:,}")
best_display['Patients Lost'] = best_display['Patients Lost'].apply(lambda x: f"{x:,}")
best_display['Retention Rate (%)'] = best_display['Retention Rate (%)'].round(1)
display(best_display)

# Bar chart: worst by patients lost (raw numbers for plotting)
fig_worst = px.bar(
    hospital_switch.head(10),
    x='PATIENTS_LOST',
    y='FIRST_HOSPITAL',
    orientation='h',
    title='Hospitals with Highest Patient Loss (first ≠ last hospital)',
    text='PATIENTS_LOST',
    color='PATIENTS_LOST',
    color_continuous_scale='Reds'
)
fig_worst.update_traces(texttemplate='%{text}', textposition='outside')
fig_worst.update_layout(height=500, margin=dict(l=150))
fig_worst.show()

### Worst Hospitals (Most Patients Lost)



,Hospital,Patients Started,Patients Lost,Retention Rate (%)
127,ULINZI PRIME HEALTH SERVICES FUND (UPHSF),"1,765",310,82.4
26,EQUITY AFIA RONGAI,737,160,78.3
0,AAR HEALTHCARE NAIROBI,211,52,75.4
64,NAIROBI WEST HOSP,165,50,69.7
57,MONALIFE PHARMACEUTICALS LTD,125,48,61.6
6,BAUS OPTICAL,112,35,68.8
116,THE AGA KHAN HOSP KIS,178,32,82.0
119,THE KAREN HOSP REFERRAL,122,27,77.9
62,NAIROBI HOSP REFERRAL,150,27,82.0
71,OPTICA LIMITED,124,25,79.8


### Best Hospitals (Most Patients Retained)



,Hospital,Patients Started,Patients Lost,Retention Rate (%)
127,ULINZI PRIME HEALTH SERVICES FUND (UPHSF),"1,765",310,82.4
26,EQUITY AFIA RONGAI,737,160,78.3
0,AAR HEALTHCARE NAIROBI,211,52,75.4
116,THE AGA KHAN HOSP KIS,178,32,82.0
64,NAIROBI WEST HOSP,165,50,69.7
62,NAIROBI HOSP REFERRAL,150,27,82.0
98,ST FRANCIS COMMUNITY KASARANI,139,23,83.5
102,ST LUKE ORTHOPEADIC ELD,139,22,84.2
57,MONALIFE PHARMACEUTICALS LTD,125,48,61.6
71,OPTICA LIMITED,124,25,79.8


### Where Do Lost Patients Go? (Sankey Diagram)

For the top 5 hospitals with the most patient losses, we visualise the flow to their new hospitals.

In [24]:
top_origins = hospital_switch.head(5)['FIRST_HOSPITAL'].tolist()
flow_data = member_summary[member_summary['FIRST_HOSPITAL'].isin(top_origins) & member_summary['SWITCHED']]
flow_sum = flow_data.groupby(['FIRST_HOSPITAL', 'LAST_HOSPITAL']).size().reset_index(name='count')
flow_sum = flow_sum.sort_values('count', ascending=False).head(15)

if not flow_sum.empty:
    labels = list(set(flow_sum['FIRST_HOSPITAL'].unique()) | set(flow_sum['LAST_HOSPITAL'].unique()))
    label_to_index = {label: i for i, label in enumerate(labels)}
    source = [label_to_index[h] for h in flow_sum['FIRST_HOSPITAL']]
    target = [label_to_index[h] for h in flow_sum['LAST_HOSPITAL']]
    value = flow_sum['count']

    fig_sankey = go.Figure(data=[go.Sankey(
        node=dict(pad=15, thickness=20, line=dict(color='black', width=0.5), label=labels),
        link=dict(source=source, target=target, value=value)
    )])
    fig_sankey.update_layout(title='Patient Flow: First Hospital → Last Hospital (Top 5 Origins)', height=600)
    fig_sankey.show()
else:
    print("Not enough flow data for Sankey diagram.")

### 7‑Day Switching (Rapid Defection)

We identify patients who switch to a different hospital within **7 days** of a previous visit. This can indicate dissatisfaction or urgent transfers.

In [25]:
# Sort by member and date
df_sorted = df.sort_values(['MEMBER NUMBER', 'ARRIVAL DATE'])

switches = []
for member, group in df_sorted.groupby('MEMBER NUMBER'):
    if len(group) < 2:
        continue
    prev_row = None
    for idx, row in group.iterrows():
        if prev_row is not None:
            if row['MAIN HOSPITAL'] != prev_row['MAIN HOSPITAL']:
                date_diff = (row['ARRIVAL DATE'] - prev_row['ARRIVAL DATE']).days
                if 0 < date_diff <= 7:
                    switches.append({
                        'MEMBER NUMBER': member,
                        'HOSPITAL_FROM': prev_row['MAIN HOSPITAL'],
                        'HOSPITAL_TO': row['MAIN HOSPITAL'],
                        'DAYS_BETWEEN': date_diff
                    })
        prev_row = row

switches_df = pd.DataFrame(switches)

if not switches_df.empty:
    # Count patients who switched from each hospital
    switch_counts = switches_df.groupby('HOSPITAL_FROM')['MEMBER NUMBER'].nunique().reset_index()
    switch_counts.columns = ['HOSPITAL', 'PATIENTS_LEFT_7D']
    
    # Total patients per hospital
    total_patients = df.groupby('MAIN HOSPITAL')['MEMBER NUMBER'].nunique().reset_index()
    total_patients.columns = ['HOSPITAL', 'TOTAL_PATIENTS']
    
    # Merge and compute rate
    switch_7d = total_patients.merge(switch_counts, on='HOSPITAL', how='left').fillna(0)
    switch_7d['SWITCH_RATE_7D'] = (switch_7d['PATIENTS_LEFT_7D'] / switch_7d['TOTAL_PATIENTS']) * 100
    switch_7d = switch_7d.sort_values('SWITCH_RATE_7D', ascending=False)
    
    # Format with commas
    display_7d = switch_7d.head(10).copy()
    display_7d['TOTAL_PATIENTS'] = display_7d['TOTAL_PATIENTS'].apply(lambda x: f"{x:,.0f}")
    display_7d['PATIENTS_LEFT_7D'] = display_7d['PATIENTS_LEFT_7D'].apply(lambda x: f"{x:.1f}")
    display_7d['SWITCH_RATE_7D'] = display_7d['SWITCH_RATE_7D'].round(1)
    
    print("### Hospitals with Highest 7‑Day Patient Defection\n")
    display(display_7d[['HOSPITAL', 'TOTAL_PATIENTS', 'PATIENTS_LEFT_7D', 'SWITCH_RATE_7D']])
else:
    print("No 7‑day switching events found.")

### Hospitals with Highest 7‑Day Patient Defection



,HOSPITAL,TOTAL_PATIENTS,PATIENTS_LEFT_7D,SWITCH_RATE_7D
7,BESTCARE HOSPITAL LIMITED,3,2.0,66.7
112,STARKEY HEARING TECHNOLOGIES LTD,8,3.0,37.5
39,KENYATTA UNIVERSITY HOSPITAL (KUTRR),25,9.0,36.0
127,THE VETERAN MISSION HOSPITALS LTD,9,3.0,33.3
15,CHIROMO LANE MEDICAL CENTRE,10,3.0,30.0
24,ELGON VIEW HOSP ELDORET,10,3.0,30.0
95,SPINE CLINIC AFRICA LTD,7,2.0,28.6
17,CONSOLATA HOSP MATHARI NYERI,52,14.0,26.9
43,LIONS SIGHT FIRST EYE,107,27.0,25.2
40,KILOME MATERNITY NURSING HOME,8,2.0,25.0


In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28478 entries, 0 to 28477
Data columns (total 48 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   EDI_NO                     0 non-null      float64       
 1   CLAIM ID                   28478 non-null  int64         
 2   CENTRAL ID                 28478 non-null  int64         
 3   CLAIM TYPE                 28478 non-null  object        
 4   SCHEME                     28478 non-null  object        
 5   MEMBER NUMBER              28478 non-null  object        
 6   INTEG MEMBER NUMBER        28477 non-null  float64       
 7   OTHER NUMBER               28478 non-null  object        
 8   OFFICE BRANCH              44 non-null     object        
 9   CARD SERIAL                28266 non-null  object        
 10  PATIENT NAME               28478 non-null  object        
 11  DOB                        28478 non-null  datetime64[ns]
 12  CAT 

## 12. Same‑Day OP‑IP Transitions

Identify members who had both Outpatient and Inpatient claims on the same day, and flag the hospital where the IP occurred.

In [27]:
# Identify same-day OP-IP events per member-date
same_day = df.groupby(['MEMBER NUMBER', 'ARRIVAL DATE']).filter(
    lambda g: set(g['SERVICE TYPE']) == {'OP', 'IP'}
)

if not same_day.empty:
    # For each event, use the hospital from the IP claim
    ip_records = same_day[same_day['SERVICE TYPE'] == 'IP'].drop_duplicates(subset=['MEMBER NUMBER', 'ARRIVAL DATE'])
    event_hospital = ip_records[['MEMBER NUMBER', 'ARRIVAL DATE', 'MAIN HOSPITAL']]

    # Count events per hospital
    hospital_counts = event_hospital.groupby('MAIN HOSPITAL').size().reset_index(name='same_day_OP_IP_events')
    hospital_counts = hospital_counts.sort_values('same_day_OP_IP_events', ascending=False)

    # Top 10
    top10 = hospital_counts.head(10).copy()
    top10['same_day_OP_IP_events'] = top10['same_day_OP_IP_events'].apply(lambda x: f"{x:,}")
    print("Top 10 Main Hospitals with Same-Day OP → IP (or both) Cases")
    print(top10.to_string(index=False))

    # Unique members per hospital
    member_counts = same_day.groupby('MAIN HOSPITAL')['MEMBER NUMBER'].nunique().reset_index(name='unique_members')
    top10_members = member_counts.sort_values('unique_members', ascending=False).head(10).copy()
    top10_members['unique_members'] = top10_members['unique_members'].apply(lambda x: f"{x:,}")
    print("\nTop 10 Hospitals by Unique Members with Same-Day OP-IP")
    print(top10_members.to_string(index=False))
else:
    print("No same-day OP-IP transitions found.")

Top 10 Main Hospitals with Same-Day OP → IP (or both) Cases
                            MAIN HOSPITAL same_day_OP_IP_events
                        NAIROBI WEST HOSP                    31
                    LIONS SIGHT FIRST EYE                    27
ULINZI PRIME HEALTH SERVICES FUND (UPHSF)                    20
           BRISTOL PARK HEALTHCARE CENTRE                    18
                 DR AGARWALS EYE HOSPITAL                    17
                    NAIROBI HOSP REFERRAL                    13
                  ST LUKE ORTHOPEADIC ELD                     9
                  THE KAREN HOSP REFERRAL                     9
              KISUMU  SPECIALIST HOSPITAL                     6
                    BISHOP KIOKO CATHOLIC                     6

Top 10 Hospitals by Unique Members with Same-Day OP-IP
                            MAIN HOSPITAL unique_members
                    LIONS SIGHT FIRST EYE             22
ULINZI PRIME HEALTH SERVICES FUND (UPHSF)             20
         

## 13. Summary & Recommendations

- **Service Type**: IP visits account for ~57% of total spend despite being only ~7% of visits. IP average cost (~124k) is much higher than OP (~7.6k).
- **Top Providers**: A small number of hospitals (UPHSF, Nairobi Hospital, Nairobi West) dominate total spending.
- **Patient Retention**: Several large hospitals lose hundreds of patients over the period. Sankey diagrams show where these patients go.
- **Rapid Switching**: Some hospitals have 7‑day defection rates >20%, suggesting potential quality or operational issues.
- **Age Analysis**: The 61-79 age bracket accounts for the highest visits, unique members, and total spend.
- **Age × Service Type**: Older adults (61-79) drive most IP spending, while OP spending is spread across age groups. Average cost per IP visit is highest in the 51-60 and 61-79 brackets.

**Next Steps**:
- Investigate root causes for high-switch hospitals (e.g., patient complaints, wait times, service gaps).
- Negotiate with high‑retention hospitals for preferred partnerships.
- Monitor monthly trends to detect sudden changes in provider market share.
- Develop preventive care programs for the high‑utilization age group (61-79).
- Review IP cost drivers for middle-aged and older adults to identify potential savings.